In [6]:
import os
import re
import pandas as pd

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)
print("✅ Library siap dan folder data/processed telah dibuat!")

✅ Library siap dan folder data/processed telah dibuat!


In [7]:
def extract_features(text, case_id):
    # 1. Ekstraksi Nomor Perkara (Lebih fleksibel menangkap pid.sus maupun pdt.sus, dengan/tanpa spasi)
    no_match = re.search(r"(?:nomor|no)[\s\:\.\-]*([\d\s\/\w\.\-]+\b(?:pid|pdt)[\w\.\s\/\\\-]+?\d{4})", text)
    no_perkara = no_match.group(1).strip() if no_match else "tidak ditemukan"
    
    # 2. Ekstraksi Tanggal Putusan (Menangkap variasi spasi pada tanggal)
    tgl_match = re.search(r"tanggal\s*[\:\-]*\s*(\d+\s+[a-z]+\s+\d{4})", text)
    tanggal = tgl_match.group(1).strip() if tgl_match else "tidak ditemukan"
    
    # 3. Ekstraksi Nama Pihak / Terdakwa
    pihak_match = re.search(r"nama\s*[\:\-]*\s*([\w\s\.\-]+?)(?=\btempat lahir\b|\bumur\b|\bjenis kelamin\b|\bkebangsaan\b)", text)
    pihak = pihak_match.group(1).strip() if pihak_match else "tidak ditemukan"
    pihak = re.sub(r"\s+", " ", pihak) 
    
    # 4. Ekstraksi Pasal (Bisa menangkap pasal dari UU Hak Cipta lama No 19/2002 maupun UU No 28/2014)
    pasal_match = re.search(r"(pasal\s+\d+\s+(?:ayat\s*\(\d+\))?.*?)(?=\bkitab\b|\buu\b|\bundang\b|\bmenimbang\b|\bttd\b)", text)
    pasal = pasal_match.group(1).strip() if pasal_match else "tidak ditemukan"
    
    # 5. Ekstraksi Ringkasan Fakta / Kronologi
    fakta_match = re.search(r"(?:didakwa|dakwaan)\s*[\:\-]*\s*(.*?)(?=\bmemperhatikan\b|\bmenuntut\b|\bmemutus\b|\bmenimbang\b)", text)
    ringkasan_fakta = fakta_match.group(1).strip()[:700] if fakta_match else text[:700]
    
    # 6. Ekstraksi Hukuman Solusi
    hukuman_match = re.search(r"penjara\s+selama\s*[\:\-]*\s*([\w\s\(\)]+)", text)
    if not hukuman_match:
        hukuman_match = re.search(r"penjara\s+selama\s+([\w\s\(\)]+)", text)
    hukuman = hukuman_match.group(1).strip() if hukuman_match else "tidak ditemukan"

    return {
        "case_id": case_id,
        "no_perkara": no_perkara,
        "tanggal": tanggal,
        "ringkasan_fakta": ringkasan_fakta,
        "pasal": pasal,
        "pihak": pihak,
        "hukuman_solusi": hukuman,
        "text_full": text
    }

print("✅ Fungsi ekstraksi fitur versi perbaikan siap digunakan.")

✅ Fungsi ekstraksi fitur versi perbaikan siap digunakan.


In [8]:
all_cases = []
files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.txt')])

for idx, filename in enumerate(files, start=1):
    file_path = os.path.join(RAW_DIR, filename)
    
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Jalankan fungsi ekstraksi
    case_data = extract_features(content, case_id=idx)
    all_cases.append(case_data)

# Konversi list ke dalam bentuk Pandas DataFrame
df = pd.DataFrame(all_cases)

# Simpan ke folder processed dalam bentuk CSV
output_csv = os.path.join(PROCESSED_DIR, "cases.csv")
df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"🎉 Sukses! Data terstruktur 37 kasus berhasil disimpan di: {output_csv}")
# Menampilkan 5 data teratas sebagai sampel preview
df.head()

🎉 Sukses! Data terstruktur 37 kasus berhasil disimpan di: data/processed\cases.csv


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,hukuman_solusi,text_full
0,1,1813 k/pid.sus/2009,03 februari 2009,bahwa ia terdakwa adiyanto pada hari sabtu tan...,pasal 72 ayat 2,a d i y a n t o.,10 sepuluh bulan dan denda rp,putusan.mahkamahagung.go.id p u t u s a n no.1...
1,2,646 k/pid.sus/2011,19 nopember 2009,kesatu bahwa ia terdakwa yurmayni pgl.may bers...,pasal 2 ayat 1 yaitu hak cipta merupakan hak e...,yurmayni pgl.may,2 dua tahun dikurangi selama terdakwa berada d...,putusan.mahkamahagung.go.id p u t u s a n nomo...
2,3,165/pid.sus/2013,11 januari 2013,penuntut umum setelah mendengar keterangan par...,pasal 72 ayat 2,tempat lahir tanggal lahir,7 tujuh bulan dikurangi selama terdakwa ditaha...,1 putusan.mahkamahagung.go.id p u t u s a n no...
3,4,882 k/pdt.sus/2012,19 september 2012,kesatu melanggar pasal 72 ayat 1 undang-undang...,pasal 72 ayat 1,tidak ditemukan,tidak ditemukan,putusan.mahkamahagung.go.id p u t u s a n nomo...
4,5,169 k/pid.sus/2015,12 februari 2011,bahwa ia terdakwa ir. andi samad pada hari sab...,pasal 72 ayat 5 jo pasal 49 ayat 3,,1 satu tahun dan membayar denda sebesar rp,putusan.mahkamahagung.go.id p u t u s a n no. ...
